# Prepare Historical Data

This notebook prepares historical climate and glacier mass balance data for the Mont Blanc study area before training the machine learning model.

The historical data includes:
- French Alps glacier mass balance from 1967–2015
- ERA5 monthly temperature from 1967–2015
- ERA5 monthly precipitation from 1967–2015

The glacier dataset will be filtered to glaciers located inside the geographic bounds of the Mont Blanc terrain model.

The final goal is to combine glacier and climate data into one yearly dataset that can be used to train and test the model.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr

print("Libraries imported successfully.")

## Load Historical Data

The first step is to set the paths to the raw glacier and climate files and make sure each file can be found before working with the data.

In [ ]:
# The notebook is inside data-science/notebooks,
# so move up one level to reach data-science.
DATA_DIR = Path("..") / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

mass_balance_file = RAW_DIR / "Bolibar2020_FrenchAlps_MassBalance_1967-2015.nc"
temperature_file = RAW_DIR / "ERA5_MontBlanc_Temperature_1967-2015.nc"
precipitation_file = RAW_DIR / "ERA5_MontBlanc_Precipitation_1967-2015.nc"

print("Mass balance:", mass_balance_file.exists())
print("Temperature:", temperature_file.exists())
print("Precipitation:", precipitation_file.exists())

## Inspect Glacier Mass Balance Data

The French Alps glacier mass balance data contains the values the model will be trained to predict. Before using it, the dataset needs to be checked for the correct years, glacier records, missing values, and data types.

The dataset will later be filtered to glaciers located within the Mont Blanc terrain study area.

**Source:** Bolibar et al. (2020), reconstructed annual glacier-wide mass balance dataset for glaciers in the French Alps.

In [ ]:
# Load the French Alps glacier mass balance dataset.
mass_balance = xr.open_dataset(mass_balance_file)

mass_balance

In [ ]:
# Inspect the glacier coordinate information.
print("Coordinates:")
print(list(mass_balance.coords))

print("\nData variables:")
print(list(mass_balance.data_vars))

In [ ]:
# Check the mass balance dataset for missing values and year coverage
print("Year range:", mass_balance["year"].min(), "-", mass_balance["year"].max())

print("\nMissing values:")
print(mass_balance.isnull().sum())

print("\nData types:")
print(mass_balance.dtypes)

In [ ]:
# Create a table containing each glacier record's ID, name, and location.
glacier_info = pd.DataFrame({
    "glacier_index": range(mass_balance.sizes["RGI_ID"]),
    "RGI_ID": mass_balance["RGI_ID"].values,
    "GLIMS_ID": mass_balance["GLIMS_ID"].values,
    "glacier_name": mass_balance["name"].values
})

# Extract longitude and latitude from the GLIMS ID.
glacier_info["longitude"] = (
    glacier_info["GLIMS_ID"].str[1:7].astype(float) / 1000
)

glacier_info["latitude"] = (
    glacier_info["GLIMS_ID"].str[8:13].astype(float) / 1000
)

glacier_info.head()

In [ ]:
# Geographic bounds of the Mont Blanc terrain model.
NORTH = 45.992668
SOUTH = 45.783327
WEST = 6.674194
EAST = 7.043610

# Keep only glaciers located inside the terrain model.
mont_blanc_glaciers = glacier_info[
    (glacier_info["latitude"] >= SOUTH) &
    (glacier_info["latitude"] <= NORTH) &
    (glacier_info["longitude"] >= WEST) &
    (glacier_info["longitude"] <= EAST)
].copy()

print("Glaciers in study area:", len(mont_blanc_glaciers))

mont_blanc_glaciers

In [ ]:
# Convert SMB data to a glacier-year table using the unique glacier index.
smb_df = pd.DataFrame({
    "glacier_index": (
        pd.Series(range(mass_balance.sizes["RGI_ID"]))
        .repeat(mass_balance.sizes["year"])
        .values
    ),
    "year": list(mass_balance["year"].values) * mass_balance.sizes["RGI_ID"],
    "mass_balance": mass_balance["SMB"].values.flatten()
})

smb_df.head()

In [ ]:
# Keep only SMB records for glaciers inside the Mont Blanc study area.
mont_blanc_smb = smb_df.merge(
    mont_blanc_glaciers,
    on="glacier_index",
    how="inner"
)

print("Rows:", len(mont_blanc_smb))
print("Glacier records:", mont_blanc_smb["glacier_index"].nunique())
print("Years:", mont_blanc_smb["year"].min(), "-", mont_blanc_smb["year"].max())

mont_blanc_smb.head()

In [116]:
# Check the Mont Blanc SMB data for missing values.
print("Missing mass balance values:", mont_blanc_smb["mass_balance"].isna().sum())

print("\nMass balance summary:")
print(mont_blanc_smb["mass_balance"].describe())

Missing mass balance values: 60

Mass balance summary:
count    2782.000000
mean       -0.710393
std         0.967558
min        -3.639968
25%        -1.402411
50%        -0.710024
75%        -0.069902
max         2.359916
Name: mass_balance, dtype: float64


In [125]:
# Verify that the missing SMB values already exist in the original NetCDF.
study_indices = mont_blanc_glaciers["glacier_index"].values

raw_study_smb = mass_balance["SMB"].values[study_indices, :]

print("Missing SMB values in original NetCDF:", pd.isna(raw_study_smb).sum())
print("Missing SMB values after reshaping:", mont_blanc_smb["mass_balance"].isna().sum())

Missing SMB values in original NetCDF: 60
Missing SMB values after reshaping: 60


In [126]:
# Remove glacier-year records that do not have an SMB target value.
mont_blanc_smb = mont_blanc_smb.dropna(
    subset=["mass_balance"]
).copy()

print("Usable SMB records:", len(mont_blanc_smb))

Usable SMB records: 2782


## Inspect Historical Temperature Data

The ERA5 temperature file contains monthly 2-meter temperature data for the Mont Blanc region from 1967–2015. The file structure, dates, coordinates, and units need to be checked before processing the values.

**Source:** ERA5 historical climate data from the Copernicus Climate Data Store.

In [151]:
# Load the ERA5 monthly temperature dataset.
temperature = xr.open_dataset(temperature_file)

temperature

precipitation = xr.open_dataset(precipitation_file)

print("Temperature longitudes:", temperature["longitude"].values)
print("Temperature latitudes:", temperature["latitude"].values)

print("Precipitation longitudes:", precipitation["longitude"].values)
print("Precipitation latitudes:", precipitation["latitude"].values)


Temperature longitudes: [6.5  6.75 7.   7.25]
Temperature latitudes: [46.   45.75]
Precipitation longitudes: [6.5  6.75 7.   7.25]
Precipitation latitudes: [46.   45.75]
